# Task 3 - Solution Design and Integration Gap Analysis
## AI-Augmented Business Analyst Capstone
**Company:** Meridian Financial | **Scenario:** FCA KYC/AML Onboarding Remediation
**Inputs:** functional_requirements.csv, requirements_backlog.csv
---
## Overview
Engineering has flagged that not all requirements can be delivered in 10 weeks with 3 engineers. James needs: (1) a requirements-to-systems mapping, (2) an integration gap analysis, (3) solution options assessment, and (4) a data model gap analysis.


## 1. Load Prior Outputs

In [ ]:
import pandas as pd
import os
os.makedirs('../outputs', exist_ok=True)

df_fr = pd.read_csv('../outputs/functional_requirements.csv')
df_nfr = pd.read_csv('../outputs/non_functional_requirements.csv')
df_backlog = pd.read_csv('../outputs/requirements_backlog.csv')

print('FRs:', len(df_fr), '| NFRs:', len(df_nfr), '| Backlog:', len(df_backlog))
print()
print(df_fr[['req_id','title','priority']].to_string(index=False))

## 2. Requirements-to-Systems Mapping

> **Activity:** Complete the mapping table. For each Sprint 1 requirement: which system implements it, what integration is needed, and what the risk is.

In [ ]:
# COMPLETE THIS TABLE
mapping_data = {
    'req_id': ['FR-001','FR-002','FR-003','FR-004','FR-005'],
    'title': [
        'Mandatory Beneficial Ownership Declaration',
        'Automated PEP and Sanctions Screening',
        'Identity Verification Evidence Capture',
        'Automated Compliance Audit Log',
        'Client Risk Rating at Application'
    ],
    'primary_system': [
        'Onboarding Microservice',
        'Onboarding Microservice',
        'Onboarding Microservice',
        'Onboarding Microservice + Audit DB',
        'Onboarding Microservice'
    ],
    'supporting_systems': [
        'Salesforce FSC, Companies House API',
        'Dow Jones API, Salesforce FSC',
        'Experian Verify API',
        'Salesforce FSC',
        'Salesforce FSC'
    ],
    'implementation_approach': ['Build','Build','Build','Build','Build'],
    'integration_risk': ['High','High','Medium','High','Medium'],
    'new_integration_required': ['Yes','Yes','No','Yes','No']
}
df_mapping = pd.DataFrame(mapping_data)
print('Requirements-to-Systems Mapping')
print('='*60)
print(df_mapping.to_string(index=False))

## 3. Integration Gap Analysis

In [ ]:
# INTEGRATION GAPS
int_gaps = {
    'int_id': ['IG-001','IG-002','IG-003','IG-004'],
    'integration': [
        'BO Declaration - Companies House API',
        'Onboarding Microservice - Salesforce FSC Sync',
        'Audit Log - Salesforce FSC',
        'Dow Jones API - PEP/Sanctions Screening'
    ],
    'current_state': [
        'Manual email-based BO lookup',
        'Basic one-way account data sync',
        'Manual CSV export from Salesforce to Compliance',
        'Manual execution by Diana in Dow Jones web interface'
    ],
    'required_state': [
        'Automated API call to Companies House in real time',
        'Bidirectional real-time sync: Salesforce FSC <-> Microservice',
        'Automated audit log entries synced to Salesforce FSC',
        'Automated API call at application submission'
    ],
    'gap_description': [
        'No API integration with Companies House. BO verification is manual and inconsistent.',
        'Microservice was not designed for bidirectional real-time sync. Requires redesign.',
        'No automated flow. Compliance team relies on manual exports for FCA reviews.',
        'No automated trigger. PEP screening depends on Diana remembering.'
    ],
    'risk': ['High','High','High','High'],
    'fallback': [
        'Manual lookup and documented telephone confirmation',
        'Operations manually creates record in Salesforce FSC within 4 hours',
        'Manual Salesforce export - acceptable for max 2 weeks during transition',
        'Diana manually executes Dow Jones check - 48-hour SLA'
    ]
}
df_int_gaps = pd.DataFrame(int_gaps)
print(df_int_gaps[['int_id','integration','risk']].to_string(index=False))

## 4. Solution Options Assessment

> **Activity:** Use Prompt 3.1 to research vendors. Complete the options tables below.

### BO Verification Options

| Option | Description | Est. Effort | Est. Cost | Risk | Verdict |
|--------|-------------|-------------|-----------|------|---------|
| A: Build In-House | Custom form + manual Companies House lookup | High | High | Low | Reject - timeline |
| B: Companies House API | Automated API call | Medium | Low | Medium | Evaluate |
| C: Third-Party BO Platform | Kyckr / Enumacloud | Low | Medium | Low | Evaluate |

### Salesforce Sync Options

| Option | Description | Est. Effort | Risk | Verdict |
|--------|-------------|-------------|------|---------|
| A: Custom Sync Service | Build bidirectional sync layer | High | High | Reject |
| B: Salesforce Out-of-the-Box | Minimal customisation | Low | High | Reject - gaps remain |
| C: MuleSoft / iPaaS | Integrate via iPaaS connector | Medium | Medium | Evaluate |

## 5. Data Model Gap Analysis

In [ ]:
# NEW DATA ENTITIES
entity_data = {
    'entity': ['BeneficialOwner','ComplianceAuditEntry','RiskRatingRecord','PEPScreeningResult'],
    'key_fields': [
        'Full name, DOB, Nationality, Address, Ownership %, Role, declaration date',
        'Timestamp, Actor, Action, Outcome, Evidence ref, Client ID, Officer ID',
        'Score (0-100), Rating (Low/Med/High), Rules applied, Review date',
        'Check date, API response, Match result, Match details, DJ ref'
    ],
    'exists_now': ['No','No','No','No'],
    'gap': [
        'No BO entity in current data model - must be created',
        'No central audit store - currently in email and Salesforce notes',
        'Risk rating not stored as entity - currently spreadsheet field',
        'PEP result not stored - Diana records in Dow Jones web only'
    ]
}
df_entities = pd.DataFrame(entity_data)
print(df_entities.to_string(index=False))

## 6. System Context Diagram

> **Activity:** Review the architecture diagram. The cell below generates it.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(16, 11))
ax.set_xlim(0, 16); ax.set_ylim(0, 11); ax.axis('off')
ax.set_title('Meridian Financial - Proposed Solution Architecture (Sprint 1)', fontsize=14, fontweight='bold', pad=15)

systems = {
    'CLIENT': (8, 9.5),
    'Web Portal\n(React)': (8, 8),
    'API Gateway': (8, 6.5),
    'Onboarding\nMicroservice': (5, 5),
    'Salesforce\nFSC': (11, 5),
    'Experian\nVerify API': (1.5, 3),
    'Dow Jones\nRisk Intel': (5, 3),
    'Companies House\nAPI': (8.5, 3),
    'Audit DB\n(PostgreSQL)': (12, 3),
}
colors_map = {
    'CLIENT': '#E8EAF6',
    'Web Portal\n(React)': '#E3F2FD',
    'API Gateway': '#E3F2FD',
    'Onboarding\nMicroservice': '#FFF9C4',
    'Salesforce\nFSC': '#E8F5E9',
    'Experian\nVerify API': '#FCE4EC',
    'Dow Jones\nRisk Intel': '#FCE4EC',
    'Companies House\nAPI': '#FCE4EC',
    'Audit DB\n(PostgreSQL)': '#E8F5E9',
}
for name, (x, y) in systems.items():
    box = FancyBboxPatch((x-1.4, y-0.7), 2.8, 1.4, boxstyle='round,pad=0.1',
                         facecolor=colors_map.get(name,'#EEE'), edgecolor='#333', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x, y, name, ha='center', va='center', fontsize=8, fontweight='bold')

arrow_props = dict(arrowstyle='->', color='#333', lw=1.3)
ax.annotate('', xy=(8, 8.7), xytext=(8, 9.2), arrowprops=arrow_props)
ax.annotate('', xy=(8, 7.1), xytext=(8, 7.4), arrowprops=arrow_props)
ax.annotate('', xy=(5.7, 5.4), xytext=(6.3, 6.3), arrowprops=arrow_props)
ax.annotate('', xy=(9.6, 5.4), xytext=(9.3, 6.3), arrowprops=arrow_props)
ax.annotate('', xy=(2.5, 3.4), xytext=(4.2, 4.4), arrowprops=arrow_props)
ax.annotate('', xy=(5.8, 3.4), xytext=(5.2, 4.4), arrowprops=arrow_props)
ax.annotate('', xy=(7.7, 3.4), xytext=(5.8, 4.4), arrowprops=arrow_props)
ax.annotate('', xy=(11.2, 3.4), xytext=(6.2, 4.4), arrowprops=arrow_props)

legend = [
    mpatches.Patch(facecolor='#E3F2FD', edgecolor='#333', label='Client-facing'),
    mpatches.Patch(facecolor='#FFF9C4', edgecolor='#333', label='Core System (Build)'),
    mpatches.Patch(facecolor='#E8F5E9', edgecolor='#333', label='Existing System'),
    mpatches.Patch(facecolor='#FCE4EC', edgecolor='#333', label='Third-Party API (Buy/Integrate)'),
]
ax.legend(handles=legend, loc='upper right', fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/solution_architecture.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ../outputs/solution_architecture.png')

## 7. Export

In [ ]:
df_mapping.to_csv('../outputs/requirements_systems_mapping.csv', index=False)
df_int_gaps.to_csv('../outputs/integration_gap_analysis.csv', index=False)
df_entities.to_csv('../outputs/data_model_gaps.csv', index=False)
print('Exported: requirements_systems_mapping.csv, integration_gap_analysis.csv, data_model_gaps.csv')

## AI Prompt Library - Task 3

### Prompt 3.1: Vendor Research
```
Research KYC platforms (Onfido, Jumio, Veriff), BO platforms (Kyckr, Enumacloud),
and PEP/sanctions alternatives to Dow Jones for a UK fintech.

Format as comparison table. For each: pricing model, FCA compliance history,
UK market presence, integration complexity.
```

### Prompt 3.2: Integration Risk Assessment
```
Current: Salesforce FSC + Onboarding Microservice (legacy) + Experian + Dow Jones.
Proposed: BO declaration form, automated PEP screening, audit log, Companies House API.

Assess: integration complexity, Salesforce sync challenges, fallback procedures,
security considerations, and testing approach.
```